# RAGAS评估框架集成
## RAGAS (Retrieval Augmented Generation Assessment)

本notebook演示RAGAS库的核心功能:
- **Faithfulness**: 声明分解 → 验证每个声明
- **AnswerRelevancy**: 从回答生成问题 → 检查与原始问题的一致性
- **ContextPrecision/Recall/Relevancy**: 检索上下文质量
- **自定义指标**: CustomAnswerCompleteness
- **SingleTurnSample** 和 **EvaluationDataset**

In [ ]:
# 安装RAGAS (如果尚未安装)
# !pip install ragas datasets langchain-openai

In [ ]:
import os
import json
from typing import List, Dict, Optional
from dataclasses import dataclass, field

# RAGAS imports
try:
    from ragas import evaluate, SingleTurnSample, EvaluationDataset
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        ContextEntityRecall,
    )
    from ragas.metrics.base import Metric, MetricWithLLM
    from ragas.llms import LangchainLLMWrapper
    RAGAS_AVAILABLE = True
    print("RAGAS 已加载")
except ImportError:
    RAGAS_AVAILABLE = False
    print("RAGAS 未安装。请运行: pip install ragas")
    
    # 提供降级实现
    print("将使用模拟实现进行演示...")

## 1. RAGAS指标模拟实现 (无需API)

以下实现了RAGAS核心指标的简化版本，便于在没有LLM API的情况下理解和测试。

In [ ]:
import re
import math
from collections import Counter

class SimFaithfulness:
    """
    Faithfulness (忠实度) 指标
    
    工作原理:
    1. 将回答分解为原子声明 (claim decomposition)
    2. 对每个声明，检查是否能从上下文中验证 (verify each claim)
    3. Faithfulness = 可验证的声明数 / 总声明数
    """
    
    @staticmethod
    def decompose_claims(answer: str) -> List[str]:
        """将回答分解为声明列表"""
        sentences = re.split(r'[.!?。！？]+', answer)
        claims = []
        for sent in sentences:
            sent = sent.strip()
            if len(sent) >= 8:
                # 对于长句，进一步分解
                if len(sent) > 120:
                    sub_parts = re.split(r'[,;，；、]+', sent)
                    claims.extend([s.strip() for s in sub_parts if len(s.strip()) >= 8])
                else:
                    claims.append(sent)
        return claims
    
    @staticmethod
    def verify_claim(claim: str, context: str) -> bool:
        """验证声明是否能从上下文中得到支持"""
        claim_words = set(re.findall(r'\w+', claim.lower()))
        context_words = set(re.findall(r'\w+', context.lower()))
        
        if not claim_words:
            return False
        
        # 关键词匹配率
        meaningful_words = {w for w in claim_words if len(w) > 2}
        if not meaningful_words:
            return False
        
        overlap = meaningful_words & context_words
        return len(overlap) / len(meaningful_words) >= 0.5
    
    @staticmethod
    def score(answer: str, context: str) -> float:
        """计算忠实度得分"""
        claims = SimFaithfulness.decompose_claims(answer)
        if not claims:
            return 0.0
        
        verified = sum(1 for c in claims if SimFaithfulness.verify_claim(c, context))
        return verified / len(claims)


class SimAnswerRelevancy:
    """
    AnswerRelevancy (回答相关性) 指标
    
    工作原理:
    1. 从回答中逆向生成可能的问题 (generate questions from answer)
    2. 计算生成的问题与原始问题的语义相似度
    3. AnswerRelevancy = mean(cosine_sim(gen_q_i, original_q))
    """
    
    @staticmethod
    def extract_key_phrases(text: str) -> List[str]:
        """从文本提取关键短语"""
        # 提取名词短语 (简化: 大写/中文关键词)
        words = re.findall(r'[A-Z][a-z]+|[一-鿿]{2,}', text)
        return words[:5]
    
    @staticmethod
    def generate_questions_from_answer(answer: str) -> List[str]:
        """从回答逆向生成问题"""
        phrases = SimAnswerRelevancy.extract_key_phrases(answer)
        questions = []
        for phrase in phrases:
            questions.append(f"什么是{phrase}？")
            questions.append(f"{phrase}有什么特点？")
        return questions[:3]  # 限制数量
    
    @staticmethod
    def jaccard_similarity(text1: str, text2: str) -> float:
        """Jaccard相似度"""
        words1 = set(re.findall(r'\w+', text1.lower()))
        words2 = set(re.findall(r'\w+', text2.lower()))
        if not words1 or not words2:
            return 0.0
        intersection = words1 & words2
        union = words1 | words2
        return len(intersection) / len(union)
    
    @staticmethod
    def score(answer: str, question: str) -> float:
        """计算回答相关性"""
        gen_questions = SimAnswerRelevancy.generate_questions_from_answer(answer)
        if not gen_questions:
            return 0.0
        
        similarities = [
            SimAnswerRelevancy.jaccard_similarity(gq, question)
            for gq in gen_questions
        ]
        return sum(similarities) / len(similarities)


class SimContextPrecision:
    """
    ContextPrecision (上下文精确度)
    
    检索到的上下文中，有多少是真正相关的？
    使用平均精度 (Average Precision) 计算。
    """
    
    @staticmethod
    def score(contexts: List[str], question: str, reference: str) -> float:
        """计算上下文精确度"""
        if not contexts:
            return 0.0
        
        question_words = set(re.findall(r'\w+', question.lower()))
        reference_words = set(re.findall(r'\w+', reference.lower()))
        relevant_words = question_words | reference_words
        
        precisions = []
        relevant_count = 0
        
        for i, ctx in enumerate(contexts, start=1):
            ctx_words = set(re.findall(r'\w+', ctx.lower()))
            overlap = ctx_words & relevant_words
            
            # 判断是否相关: 足够多的关键词重叠
            is_relevant = len(overlap) > 0 and len(overlap) / max(1, len(relevant_words)) > 0.1
            
            if is_relevant:
                relevant_count += 1
                precisions.append(relevant_count / i)
        
        if not precisions:
            return 0.0
        
        return sum(precisions) / len(precisions)


class SimContextRecall:
    """
    ContextRecall (上下文召回率)
    
    参考答案中的信息有多少被检索上下文覆盖？
    """
    
    @staticmethod
    def score(contexts: List[str], reference: str) -> float:
        """计算上下文召回率"""
        if not contexts or not reference:
            return 0.0
        
        # 将参考答案分解为句子
        ref_sentences = [s.strip() for s in re.split(r'[.!?。！？]+', reference) if len(s.strip()) > 5]
        if not ref_sentences:
            return 0.0
        
        # 合并所有上下文
        full_context = ' '.join(contexts)
        context_words = set(re.findall(r'\w+', full_context.lower()))
        
        # 对每个参考答案句子，检查上下文覆盖
        covered_count = 0
        for sent in ref_sentences:
            sent_words = set(re.findall(r'\w+', sent.lower()))
            if not sent_words:
                continue
            overlap = sent_words & context_words
            if len(overlap) / len(sent_words) >= 0.3:
                covered_count += 1
        
        return covered_count / len(ref_sentences)


class SimContextRelevancy:
    """
    ContextRelevancy (上下文相关性)
    
    检索到的上下文与问题的相关程度。
    """
    
    @staticmethod
    def score(contexts: List[str], question: str) -> float:
        """计算上下文相关性"""
        if not contexts:
            return 0.0
        
        question_words = set(re.findall(r'\w+', question.lower()))
        if not question_words:
            return 0.0
        
        relevancy_scores = []
        for ctx in contexts:
            ctx_words = set(re.findall(r'\w+', ctx.lower()))
            overlap = question_words & ctx_words
            relevancy_scores.append(len(overlap) / len(question_words))
        
        return sum(relevancy_scores) / len(relevancy_scores)


print("RAGAS模拟指标实现完成")
print("可用指标: Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, ContextRelevancy")

## 2. 测试数据准备

In [ ]:
# 构造测试样本
test_samples = [
    {
        "question": "什么是RAG技术？",
        "answer": "RAG（检索增强生成）是一种结合信息检索和文本生成的AI技术。"
                  "它从知识库中检索相关文档，然后将这些文档作为上下文提供给LLM，"
                  "以生成更准确、更可靠的回答。RAG能有效减少模型的幻觉现象。",
        "contexts": [
            "RAG (Retrieval-Augmented Generation) 是一种结合了信息检索和文本生成的人工智能架构。"
            "它首先从知识库中检索相关文档，然后将检索结果注入到大语言模型的提示中。",
            "这种方法可以显著减少大语言模型的幻觉现象，提高回答的事实准确性。"
        ],
        "reference": "RAG是一种AI架构，通过结合检索系统和生成模型来提高答案准确性。"
                     "它从外部知识库检索信息，然后将信息提供给LLM生成回答，减少幻觉现象。",
    },
    {
        "question": "深度学习的主要挑战有哪些？",
        "answer": "深度学习面临的主要挑战包括：1）需要大量标注数据；2）计算资源消耗大；"
                  "3）模型可解释性差；4）容易过拟合。此外，深度学习模型容易犯罪。",  # 最后一句是幻觉
        "contexts": [
            "深度学习的主要挑战包括：对大规模标注数据的需求、高昂的计算成本、"
            "模型的黑箱特性导致可解释性差，以及在数据不足时容易过拟合。",
            "深度学习在图像识别领域取得突破性进展，AlphaGo战胜了人类围棋冠军。"  # 不太相关的上下文
        ],
        "reference": "深度学习面临数据需求大、计算成本高、可解释性差和过拟合等挑战。",
    },
    {
        "question": "Python和Java有什么区别？",
        "answer": "Python是动态类型语言，语法简洁，适合快速开发。Python有丰富的科学计算库。"
                  "Python易于学习，社区活跃。",
        "contexts": [
            "Python是一种解释型、动态类型的编程语言，以其简洁的语法和丰富的库生态系统闻名。",
            "Java是一种编译型、静态类型的编程语言，以其平台独立性和强大的企业级生态系统著称。",
            "Python和Java的主要区别包括：类型系统（动态vs静态）、执行方式（解释vs编译）、"
            "语法风格（简洁vs冗长）、生态系统重点（数据科学vs企业应用）。"
        ],
        "reference": "Python是动态类型解释型语言，语法简洁，适合数据科学；"
                     "Java是静态类型编译型语言，适合企业级应用。",
    },
]

print(f"准备了 {len(test_samples)} 个测试样本")

## 3. 使用模拟指标评估所有样本

In [ ]:
def evaluate_sample(sample):
    """对单个样本运行所有评估指标"""
    results = {}
    
    # 1. Faithfulness
    results['faithfulness'] = SimFaithfulness.score(
        sample['answer'], ' '.join(sample['contexts'])
    )
    
    # 2. AnswerRelevancy
    results['answer_relevancy'] = SimAnswerRelevancy.score(
        sample['answer'], sample['question']
    )
    
    # 3. ContextPrecision
    results['context_precision'] = SimContextPrecision.score(
        sample['contexts'], sample['question'], sample['reference']
    )
    
    # 4. ContextRecall
    results['context_recall'] = SimContextRecall.score(
        sample['contexts'], sample['reference']
    )
    
    # 5. ContextRelevancy
    results['context_relevancy'] = SimContextRelevancy.score(
        sample['contexts'], sample['question']
    )
    
    return results


# 评估所有样本
all_results = []
for i, sample in enumerate(test_samples):
    result = evaluate_sample(sample)
    all_results.append(result)
    print(f"\n样本 {i+1}: {sample['question'][:40]}...")
    print(f"  声明数: {len(SimFaithfulness.decompose_claims(sample['answer']))}")
    for metric, score in result.items():
        bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
        print(f"  {metric:20s}: {score:.3f} {bar}")

# 计算平均值
print(f"\n{'='*50}")
print("平均指标得分:")
for metric in all_results[0].keys():
    avg = sum(r[metric] for r in all_results) / len(all_results)
    print(f"  {metric:20s}: {avg:.3f}")

## 4. 自定义指标: AnswerCompleteness (回答完整性)

检查回答是否涵盖了问题的所有方面。
对于多问题查询，检查每个子问题是否都被回答。

In [ ]:
class CustomAnswerCompleteness:
    """
    自定义回答完整性指标
    
    评估回答是否完整覆盖了问题提出的所有要点。
    
    方法:
    1. 解析问题中的子问题/要点
    2. 对每个子问题，检查回答是否包含了相关信息
    3. 完整性 = 被覆盖的子问题数 / 总子问题数
    """
    
    @staticmethod
    def extract_sub_questions(question: str) -> List[str]:
        """从问题中提取子问题"""
        # 按问号和分号分割
        sub_questions = re.split(r'[?？;；]', question)
        sub_questions = [q.strip() for q in sub_questions if len(q.strip()) > 5]
        
        if len(sub_questions) <= 1:
            # 对于单个问题，寻找要点关键词
            list_items = re.findall(r'[1-9一二三四五六七八九][.、）\)]', question)
            if list_items:
                # 有编号列表，按编号分割
                parts = re.split(r'[1-9一二三四五六七八九][.、）\)]', question)
                sub_questions = [p.strip() for p in parts if len(p.strip()) > 3]
            else:
                sub_questions = [question]
        
        return sub_questions
    
    @staticmethod
    def check_coverage(sub_question: str, answer: str) -> float:
        """检查子问题是否被回答覆盖"""
        sq_words = set(re.findall(r'\w+', sub_question.lower()))
        ans_words = set(re.findall(r'\w+', answer.lower()))
        
        if not sq_words:
            return 0.0
        
        # 去除常见词后的关键词
        stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how', 
                      'why', 'when', 'where', 'who', 'which', '的', '了', '是', '在'}
        meaningful_words = sq_words - stop_words
        
        if not meaningful_words:
            return 0.5  # 无法判断
        
        overlap = meaningful_words & ans_words
        return len(overlap) / len(meaningful_words)
    
    @staticmethod
    def score(question: str, answer: str) -> float:
        """计算回答完整性"""
        sub_questions = CustomAnswerCompleteness.extract_sub_questions(question)
        if not sub_questions:
            return 0.0
        
        coverages = [
            CustomAnswerCompleteness.check_coverage(sq, answer)
            for sq in sub_questions
        ]
        return sum(coverages) / len(coverages)


# 测试完整性指标
multi_questions = [
    "RAG的优点是什么？它的工作原理是怎样的？它有哪些局限性？",
    "请比较Python和Java：1）类型系统 2）性能 3）生态系统 4）学习曲线",
]

test_answer = "RAG的优点包括减少幻觉和实时更新知识。它的原理是先检索后生成。" \
              "Python是动态类型，学习简单。Java是静态类型，性能较好。" \
              "Python有丰富的库，Java企业生态成熟。Python学习曲线平缓。"

for q in multi_questions:
    sub_qs = CustomAnswerCompleteness.extract_sub_questions(q)
    score = CustomAnswerCompleteness.score(q, test_answer)
    print(f"\n问题: {q[:60]}...")
    print(f"  子问题: {sub_qs}")
    print(f"  完整性得分: {score:.3f}")

## 5. 使用真实RAGAS (如果已安装)

以下代码演示如何在有API访问的情况下使用真正的RAGAS库。

In [ ]:
if RAGAS_AVAILABLE:
    # 初始化RAGAS指标 (需要LLM)
    # 注意: 需要提供LLM实例
    
    # 创建SingleTurnSample
    sample = SingleTurnSample(
        user_input=test_samples[0]['question'],
        response=test_samples[0]['answer'],
        retrieved_contexts=test_samples[0]['contexts'],
        reference=test_samples[0]['reference'],
    )
    
    print("SingleTurnSample 创建完成:")
    print(f"  用户输入: {sample.user_input[:50]}...")
    print(f"  检索上下文数: {len(sample.retrieved_contexts)}")
    
    # 创建EvaluationDataset
    dataset = EvaluationDataset.from_list([
        {
            "user_input": s['question'],
            "response": s['answer'],
            "retrieved_contexts": s['contexts'],
            "reference": s['reference'],
        }
        for s in test_samples
    ])
    
    print(f"\nEvaluationDataset 创建完成:")
    print(f"  样本数: {len(dataset)}")
    
    # 评估 (需要LLM，这里只是演示结构)
    print("\n要运行真实评估，需要:")
    print("  1. 配置LLM: from ragas.llms import LangchainLLMWrapper")
    print("  2. 创建指标: Faithfulness(), AnswerRelevancy(), ContextPrecision()")
    print("  3. 运行: result = evaluate(dataset, metrics=[...])")
else:
    print("RAGAS未安装。上面的模拟实现提供了相同的概念。")
    print("\n安装RAGAS:")
    print("  pip install ragas langchain-openai")

## 6. 指标对比分析

比较不同样本的指标表现，识别系统的强弱项。

In [ ]:
# 汇总所有指标
import json

summary = {
    "total_samples": len(test_samples),
    "per_sample": [],
    "averages": {},
    "strengths": [],
    "weaknesses": [],
}

for i, (sample, result) in enumerate(zip(test_samples, all_results)):
    summary["per_sample"].append({
        "sample_id": i + 1,
        "question": sample["question"][:50],
        "metrics": result,
    })

# 计算平均值
for metric in all_results[0].keys():
    avg = sum(r[metric] for r in all_results) / len(all_results)
    summary["averages"][metric] = round(avg, 4)

# 识别强弱项
sorted_metrics = sorted(summary["averages"].items(), key=lambda x: x[1])
summary["weaknesses"] = [m for m, s in sorted_metrics[:2]]
summary["strengths"] = [m for m, s in sorted_metrics[-2:]]

print("指标对比分析:")
print(f"\n优点 (最高分指标): {', '.join(summary['strengths'])}")
print(f"短板 (最低分指标): {', '.join(summary['weaknesses'])}")

print(f"\n完整汇总 (JSON):")
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 7. 总结

本notebook演示了RAGAS评估框架的核心概念:

1. **Faithfulness (忠实度)**: 回答是否基于检索到的上下文
2. **AnswerRelevancy (回答相关性)**: 回答是否针对问题
3. **ContextPrecision (上下文精确度)**: 检索结果是否精确
4. **ContextRecall (上下文召回率)**: 检索结果是否完整
5. **ContextRelevancy (上下文相关性)**: 上下文是否与问题相关
6. **Custom Metric (自定义指标)**: 根据需求扩展评估维度

关键概念:
- **SingleTurnSample**: 单轮对话的评估样本
- **EvaluationDataset**: 评估数据集，包含多个样本
- **声明分解** → **逐声明验证**: 忠实度评估的核心思路
- **逆向问题生成** → **相似度比对**: 相关性评估的核心思路